# NAS当前源到研究价格层
兼容canonical_none_refresh_v1，显式绑定工作区和原始/复权事件哈希；校验真实会话收盘和完成标志。保留legacy格式。新原生点位与旧冻结provider重叠比较，不重算模型或回测。

In [1]:
import os,json,qlib
from pathlib import Path
os.environ['MLFLOW_ALLOW_FILE_STORE']='true'
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/nas_current_point_marks_20261014'
qlib.init(provider_uri=str(root/'data/qlib/full_path_tiered_allocation_20261013'),region='us',kernels=1)
from src.workflow.nas_current_point_marks import run
summary=run(root=root,card=card,plan=json.loads((card/'plan.json').read_text()))
summary

{'experiment': 'nas_current_point_marks_20261014',
 'plan_sha256': 'ab53b736525e4ba96ddd203d4334e6a0215df96d0e0047ab548fcd080722f772',
 'decision': 'canonical_core_price_layer_ready_context_lagged',
 'objective_achieved': False,
 'new_fits': 0,
 'new_backtests': 0,
 'source_id': '149ef504c382386397944b1e7af203b3eace9d2c9e43e09d9164c9a3691b302c',
 'source_schema': 'canonical_none_refresh_v1',
 'core_as_of': '2026-10-13',
 'context_as_of': '2026-10-09',
 'all_model_inputs_updated': False,
 'native_price_rows': 16080,
 'overlap_price_rows': 16064,
 'new_price_rows': 16,
 'maximum_overlap_price_difference': {'$execution_price': 0.0,
  '$raw_execution_price': 0.0},
 'overlap_prices_identical': True,
 'required_points_per_symbol_session': 4,
 'observed_session_close_required': True,
 'unfinished_required_marks_rejected': True,
 'frozen_reference_provider_unchanged': True,
 'provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/nas_current_point_marks_20261014',
 'region': 'us',
 'as_of': '2

# NAS刷新源接入研究价格层

实际复现了旧加载器对`canonical_none_refresh_v1`抛出`KeyError: complete`的问题。旧函数只接受最初vendor采集清单，而日常NAS刷新已使用规范化NONE缓存。因此“数据已补到NAS”不等于该研究加载器已能消费它。

修复：保留`semis_minute_source_v1`，新增canonical格式分支，显式要求workspace_root，核验清单、原始缓存及复权事件哈希，拒绝路径越界、重复分钟和截止日之外的数据；研究工作流调用点同步传入root。没有把新格式伪装成旧原始vendor文件。

同时收紧价格时点合同：按真实XNYS会话收盘查找结束点（含半日市），不再将当天最后一条现有K线当作收盘。所需节点必须存在、OHLC有效且is_final标志通过；UTC存储先转换到纽约时区再写入原生时间。该标志仍只是采集方完成判断，不证明永不修订或真实首收时刻。

## 实际新源读回

- NAS source_id：`149ef504c382386397944b1e7af203b3eace9d2c9e43e09d9164c9a3691b302c`。
- 核心区间：2018-10-12至2026-10-13；每资产每会话4个研究点：09:45已知收盘、09:47开盘成交、10:32恢复成交与实际会话收盘。
- 新原生价格行16080，旧冻结重叠16064，新增16行（10月12–13日、两资产、四时点）。这里是稀疏价格节点，不是新增16条原始分钟，更不代表完整分钟风险监测。
- 原始股价与研究复权价格的最大重叠差均为0，旧冻结provider未改动。来源范围扩展不自动重算旧模型或回测。
- 同行上下文仍截至10月9日，`all_model_inputs_updated=false`；本卡仅使核心价格层可接入新NAS格式，不能宣称所有模型输入都同步到13日。

测试覆盖两种清单在普通日/半日市的原生结果一致、canonical UTC时间正确转换、原始哈希篡改拒绝，以及只有午间最后记录而缺真实收盘时拒绝。初次测试夹具缺复权表必需列，修正为空但有正确列的夹具后通过；没有为通过测试放宽数据合同。4项相关检查通过。

决定：**canonical_core_price_layer_ready_context_lagged**。0新拟合、0回测，未宣称收益改善；稳定净SOXX超额目标未完成，SIM模型与策略参数未修改。

下一步仍需将必要的同行和日频特征按同一来源/可用时点对齐，保留未知，不靠截短或填值伪装共同新样本。完成输入资格后才能开展新的冻结研究；已失败路线不因数据适配完成而重新调参。

[新节点](new_marks.csv)、[重叠核对](overlap_audit.csv)、[对象清单](verification_manifest.json)。Notebook、原生读回和NAS证据回读通过，原始大数据仍由既有NAS快照持久保存。源数据的历史OHLC异常、回溯复权、真实分红/成交及首发资格限制没有被本适配消除。
